# Signal to Risk to Backtest Step Walkthrough

This notebook is a glass-box inspection tool for the current package code.

It walks one real slice of data through these layers:

- Alpaca historical bars
- `TradingRuntimeSpec`
- strategy signal generation
- risk policy evaluation
- backtest step accounting

The goal is to make the system explainable without duplicating the implementation.

In [ ]:
from __future__ import annotations

import sys
from dataclasses import asdict
from pathlib import Path

import pandas as pd
from IPython.display import display

repo_root = Path.cwd().resolve().parents[1]
src_path = repo_root / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

repo_root

In [ ]:
from securities_analysis.backtest.costs import ExecutionCostModel
from securities_analysis.backtest.engine import StrategyBacktester
from securities_analysis.config import load_alpaca_settings
from securities_analysis.execution.alpaca import AlpacaTrader
from securities_analysis.runtime import RiskSpec, StrategySpec, TradingRuntimeSpec

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 160)

## Choose a simple inspection setup

Start with one liquid symbol and one modest date range so the walkthrough stays easy to inspect.

In [ ]:
cfg_path = repo_root / 'data' / 'alpaca_keys.cfg'
symbol = 'SPY'
asset_class = 'equity'
freq = 'day'
start = '2024-08-01'
end = '2024-12-31'

periods_per_year = 252
initial_equity = 100_000.0
spread_bps = 1.0
session_return = 0.0

## Build the runtime spec

This is the shared configuration object that keeps research and paper trading aligned.

In [ ]:
strategy_spec = StrategySpec(
    family='trend',
    lookback_bars=30,
    vol_lookback_bars=10,
    target_volatility=0.10,
    max_gross_leverage=1.0,
    allow_short=False,
)

risk_spec = RiskSpec(
    periods_per_year=periods_per_year,
    max_gross_leverage=1.0,
    max_position_notional_pct=0.10,
    max_trade_notional_pct=0.10,
    max_daily_drawdown_pct=0.03,
    max_spread_bps=20.0,
    fractional_kelly=0.10,
    max_kelly_fraction=0.25,
    allow_short=False,
)

runtime_spec = TradingRuntimeSpec(
    symbol=symbol,
    asset_class=asset_class,
    periods_per_year=periods_per_year,
    strategy=strategy_spec,
    risk=risk_spec,
)

runtime_spec.describe()

In [ ]:
pd.json_normalize(runtime_spec.to_dict()).T.rename(columns={0: 'value'})

## Load bars from Alpaca

In [ ]:
settings = load_alpaca_settings(cfg_path=cfg_path)
trader = AlpacaTrader(settings)

bars = trader.get_historical_bar_objects(
    ticker=symbol,
    start=start,
    end=end,
    freq=freq,
    asset_class=asset_class,
)

len(bars), bars[:2], bars[-2:]

In [ ]:
bars_frame = pd.DataFrame([
    {
        'timestamp': bar.end_time,
        'open': bar.open_price,
        'high': bar.high_price,
        'low': bar.low_price,
        'close': bar.close_price,
        'volume': bar.volume,
    }
    for bar in bars
])

bars_frame.tail(10)

## Walk one bar through the strategy layer

We warm the strategy on all prior bars and then inspect the signal created on the final bar.

In [ ]:
strategy = runtime_spec.strategy.build(symbol=symbol, periods_per_year=periods_per_year)

history_bars = bars[:-1]
focus_bar = bars[-1]

for bar in history_bars:
    strategy.on_bar(bar)

signal = strategy.on_bar(focus_bar)
signal

In [ ]:
signal_frame = pd.json_normalize(asdict(signal), sep='.')
signal_frame.T.rename(columns={0: 'value'})

Key strategy ideas to watch here:

- `target_position`: desired directional exposure from the signal layer
- `confidence`: rough signal strength
- `metadata.momentum_score`: recent cumulative momentum
- `metadata.normalized_momentum`: momentum scaled by noise
- `metadata.realized_volatility`: recent observed volatility used for vol targeting

## Run the risk layer on that signal

In [ ]:
risk_policy = runtime_spec.risk.build()

decision = risk_policy.evaluate(
    signal=signal,
    equity=initial_equity,
    price=focus_bar.close_price,
    spread_bps=spread_bps,
    session_return=session_return,
    historical_strategy_returns=strategy.get_strategy_returns(),
)

decision

In [ ]:
decision_frame = pd.json_normalize(asdict(decision), sep='.')
decision_frame.T.rename(columns={0: 'value'})

Key risk ideas to watch here:

- `approved`: whether risk allows the trade
- `desired_notional`: target dollars to allocate
- `desired_quantity`: target shares implied by that notional
- `metadata.target_fraction`: final fraction of equity after risk sizing
- `metadata.kelly_fraction`: conservative Kelly estimate used in sizing

## Translate the decision into an order intent

For this notebook we assume the current position is flat, so the desired quantity is also the order delta.

In [ ]:
current_quantity = 0.0
desired_quantity_delta = decision.desired_quantity - current_quantity
order_intent = risk_policy.build_order_intent(
    signal=signal,
    desired_quantity_delta=desired_quantity_delta,
    time_in_force='day',
)

order_intent

In [ ]:
if order_intent is None:
    print('No order intent produced.')
else:
    display(pd.json_normalize(asdict(order_intent), sep='.').T.rename(columns={0: 'value'}))

## Run the full backtester on the same bars

This lets us compare the single-path walkthrough with the final recorded backtest step.

In [ ]:
backtester = StrategyBacktester(
    strategy=runtime_spec.strategy.build(symbol=symbol, periods_per_year=periods_per_year),
    risk_policy=runtime_spec.risk.build(),
    initial_equity=initial_equity,
    spread_bps=spread_bps,
    cost_model=ExecutionCostModel(
        commission_bps=0.0,
        slippage_bps=2.0,
        market_impact_bps_per_turnover=5.0,
    ),
)

result = backtester.run(bars)
result

In [ ]:
steps_frame = pd.DataFrame([
    {
        'timestamp': step.bar.end_time,
        'close_price': step.bar.close_price,
        'approved': step.approved,
        'reason': step.reason,
        'target_position': step.target_position,
        'desired_quantity': step.desired_quantity,
        'equity': step.equity,
        'session_return': step.session_return,
        'spread_bps': step.spread_bps,
        'turnover_fraction': step.turnover_fraction,
        'gross_return': step.gross_return,
        'cost_return': step.cost_return,
        'net_return': step.net_return,
    }
    for step in result.steps
])

steps_frame.tail(5)

In [ ]:
final_step = result.steps[-1]
pd.json_normalize(asdict(final_step), sep='.').T.rename(columns={0: 'value'})

## Interpretation checklist

When this notebook is working, you should be able to answer:

1. Why did the strategy want `target_position` of this size?
2. Why did the risk layer approve or reject it?
3. How did `desired_notional` turn into `desired_quantity`?
4. How did that decision show up in the recorded backtest step?
5. How did transaction costs reduce `gross_return` to `net_return`?

If any of those still feel opaque, that is the next place to deepen the validation tooling.